# OmniCare Clinical Copilot — Full Pipeline

**Complete patient journey in a single notebook:**
1. **Consultation** — Audio → Whisper ASR → Transcript → MedGemma → SOAP Note
2. **Admission** — Synthea FHIR → Vitals/Conditions → MedGemma → Admission Note
3. **Radiology** — Medical Image → MedGemma Multimodal → Radiology Report
4. **Discharge** — Aggregate All → MedGemma → Comprehensive Discharge Summary

**Models:** `google/medgemma-1.5-4b-it` (4-bit) + `openai/whisper-large-v3-turbo` — fits in T4 16GB (~6GB VRAM)

**Prerequisites:**
- Colab with T4 GPU (Runtime → Change runtime type → T4)
- Colab Secrets (Key icon in sidebar): `GITHUB_PAT` + `HF_TOKEN`

---
## Section 0: Colab Bootstrap & Dependencies

In [ ]:
# ===========================================================
# Colab Bootstrap — clone repo & set up paths
# ===========================================================
import os, sys

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_DIR = '/content/omnicare-clinical-copilot'

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        try:
            token = userdata.get('GITHUB_PAT')
        except Exception:
            raise RuntimeError(
                'GITHUB_PAT secret not found. Go to the Key icon in the left sidebar → '
                'Add a secret named GITHUB_PAT with your GitHub Personal Access Token.'
            )
        repo_url = f'https://{token}@github.com/Yashground/omnicare-clinical-copilot.git'
        ret = os.system(f'git clone {repo_url} {REPO_DIR}')
        if ret != 0 or not os.path.exists(REPO_DIR):
            raise RuntimeError(
                f'git clone failed (exit code {ret}). Check your GITHUB_PAT token '
                'and that it has repo access.'
            )
    NOTEBOOKS_DIR = os.path.join(REPO_DIR, 'notebooks')
else:
    # Local: try __file__ first, fall back to cwd
    try:
        NOTEBOOKS_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        NOTEBOOKS_DIR = os.getcwd()
    # If running from repo root, adjust to notebooks/
    if os.path.isdir(os.path.join(NOTEBOOKS_DIR, 'notebooks', 'utils')):
        NOTEBOOKS_DIR = os.path.join(NOTEBOOKS_DIR, 'notebooks')

# Verify utils/ package exists before adding to sys.path
_utils_init = os.path.join(NOTEBOOKS_DIR, 'utils', '__init__.py')
assert os.path.isfile(_utils_init), (
    f'utils/ package not found at {NOTEBOOKS_DIR}/utils/. '
    'Ensure the repo cloned correctly and contains notebooks/utils/__init__.py.'
)

if NOTEBOOKS_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOKS_DIR)

os.makedirs('/content/encounters', exist_ok=True)
os.makedirs('/content/sample_data', exist_ok=True)

print(f'Environment: {"Colab" if IN_COLAB else "Local"} | Notebooks: {NOTEBOOKS_DIR}')
print(f'utils/ package: FOUND at {_utils_init}')

In [ ]:
# Install dependencies
%pip install -q transformers accelerate bitsandbytes torch
%pip install -q soundfile librosa
%pip install -q pydicom Pillow
%pip install -q fhir.resources pydantic
%pip install -q huggingface_hub

In [ ]:
# HuggingFace login for gated MedGemma access
from huggingface_hub import login

if IN_COLAB:
    try:
        hf_token = userdata.get('HF_TOKEN')
        login(token=hf_token)
        print('Logged in via Colab secret HF_TOKEN')
    except Exception:
        print('HF_TOKEN secret not found. Falling back to interactive login.')
        login()
else:
    login()

---
## Section 1: Load Models

In [ ]:
import torch

print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Runtime → Change runtime type → T4 GPU')

In [ ]:
# Load MedGemma 1.5-4b-it (4-bit quantized)
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MEDGEMMA_MODEL_ID = 'google/medgemma-1.5-4b-it'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f'Loading {MEDGEMMA_MODEL_ID} (4-bit)...')
medgemma_model = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto'
)
medgemma_processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID)
print(f'MedGemma loaded. VRAM: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB')

In [ ]:
# Load Whisper Large v3 Turbo
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

WHISPER_MODEL_ID = 'openai/whisper-large-v3-turbo'

print(f'Loading {WHISPER_MODEL_ID}...')
whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL_ID, torch_dtype=torch.float16, low_cpu_mem_usage=True
).to('cuda')
whisper_processor = AutoProcessor.from_pretrained(WHISPER_MODEL_ID)

asr_pipeline = pipeline(
    'automatic-speech-recognition',
    model=whisper_model,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    torch_dtype=torch.float16,
    device='cuda'
)
print(f'Whisper loaded. Total VRAM: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB')

In [ ]:
# Quick sanity check
test_msgs = [{'role': 'user', 'content': 'What are the common symptoms of pneumonia?'}]
inputs = medgemma_processor.apply_chat_template(
    test_msgs, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors='pt'
).to(medgemma_model.device)
with torch.no_grad():
    out = medgemma_model.generate(**inputs, max_new_tokens=150)
new_tokens = out[0][inputs['input_ids'].shape[1]:]
print('MedGemma sanity check:')
print(medgemma_processor.decode(new_tokens, skip_special_tokens=True)[:300])

In [ ]:
!nvidia-smi

---
## Section 2: Consultation — Audio → SOAP Note

In [ ]:
from utils.encounter_state import new_encounter_id, blank_encounter, save_encounter, update_stage
from utils.prompts import WHISPER_MEDICAL_PROMPT, SOAP_SYSTEM_PROMPT, SOAP_USER_TEMPLATE
from utils.mcp_tools import transcribe_audio, generate_soap_note, parse_soap_sections

# Create encounter
encounter_id = new_encounter_id()
encounter = blank_encounter(
    encounter_id=encounter_id,
    patient_name='John Smith',
    mrn='MRN-2026-001',
    dob='1965-03-15'
)
save_encounter(encounter)
print(f'Encounter created: {encounter_id}')

In [ ]:
# Audio input — set USE_MOCK_TEXT=False and provide audio_path to use real audio
USE_MOCK_TEXT = True
audio_path = None  # e.g., '/content/sample_data/consultation.wav'

MOCK_TRANSCRIPT = (
    'Patient presents with a 3-day history of productive cough with yellowish sputum. '
    'Patient also reports low-grade fever peaking at 100.4 degrees Fahrenheit and fatigue. '
    'No known drug allergies. Past medical history significant for type 2 diabetes mellitus '
    'controlled with metformin 500mg twice daily and hypertension managed with lisinopril 10mg daily. '
    'On examination, temperature 99.8 F, blood pressure 138 over 82, heart rate 88 beats per minute, '
    'respiratory rate 18, SpO2 96 percent on room air. Lung auscultation reveals crackles in the '
    'right lower lobe. I will prescribe amoxicillin 500mg three times daily for 7 days. '
    'Ordering a chest X-ray to rule out pneumonia. CBC and BMP labs ordered. '
    'Patient advised to return if symptoms worsen or fever exceeds 101.5 F. '
    'Follow up in one week.'
)

if USE_MOCK_TEXT:
    transcript = MOCK_TRANSCRIPT
    print('Using mock transcript.')
else:
    print(f'Transcribing: {audio_path}')
    transcript = transcribe_audio(audio_path, asr_pipeline, WHISPER_MEDICAL_PROMPT)

print(f'\nTranscript ({len(transcript)} chars):\n{transcript}')

In [ ]:
# Generate SOAP note
print('Generating SOAP note with MedGemma...')
soap_raw = generate_soap_note(transcript, medgemma_model, medgemma_processor, max_new_tokens=1024)

print('\n' + '='*60)
print('SOAP NOTE')
print('='*60)
print(soap_raw)

In [ ]:
# Parse and save
soap_sections = parse_soap_sections(soap_raw)
for section, content in soap_sections.items():
    print(f'{section.upper()}: {"YES" if content.strip() else "EMPTY"} ({len(content)} chars)')

encounter = update_stage(encounter_id, 'consultation', {
    'audio_file': audio_path,
    'transcript': transcript,
    'soap_note': soap_sections
})
print(f'\nConsultation saved to encounter {encounter_id}')

---
## Section 3: Admission — FHIR Vitals → Admission Note

In [ ]:
import json
from utils.fhir_helpers import (
    parse_fhir_bundle, extract_vitals, extract_conditions,
    extract_medications, extract_patient_demographics, format_vitals_summary
)
from utils.mcp_tools import generate_admission_note

# Sample FHIR Bundle (embedded for demo — replace with Synthea output)
SAMPLE_BUNDLE = {
    'resourceType': 'Bundle', 'type': 'collection',
    'entry': [
        {'resource': {'resourceType': 'Patient', 'id': 'patient-001',
                      'name': [{'given': ['John'], 'family': 'Smith'}],
                      'birthDate': '1965-03-15', 'gender': 'male',
                      'address': [{'line': ['123 Main St'], 'city': 'Boston', 'state': 'MA', 'postalCode': '02101'}]}},
        {'resource': {'resourceType': 'Observation', 'status': 'final',
                      'category': [{'coding': [{'code': 'vital-signs'}]}],
                      'code': {'coding': [{'code': '8310-5', 'display': 'Body temperature'}]},
                      'valueQuantity': {'value': 99.8, 'unit': 'degF'},
                      'effectiveDateTime': '2026-03-25T08:00:00Z'}},
        {'resource': {'resourceType': 'Observation', 'status': 'final',
                      'category': [{'coding': [{'code': 'vital-signs'}]}],
                      'code': {'coding': [{'code': '85354-9', 'display': 'Blood pressure panel'}]},
                      'component': [
                          {'code': {'coding': [{'display': 'Systolic'}]}, 'valueQuantity': {'value': 138, 'unit': 'mmHg'}},
                          {'code': {'coding': [{'display': 'Diastolic'}]}, 'valueQuantity': {'value': 82, 'unit': 'mmHg'}}
                      ],
                      'effectiveDateTime': '2026-03-25T08:00:00Z'}},
        {'resource': {'resourceType': 'Observation', 'status': 'final',
                      'category': [{'coding': [{'code': 'vital-signs'}]}],
                      'code': {'coding': [{'code': '8867-4', 'display': 'Heart rate'}]},
                      'valueQuantity': {'value': 88, 'unit': 'bpm'},
                      'effectiveDateTime': '2026-03-25T08:00:00Z'}},
        {'resource': {'resourceType': 'Observation', 'status': 'final',
                      'category': [{'coding': [{'code': 'vital-signs'}]}],
                      'code': {'coding': [{'code': '9279-1', 'display': 'Respiratory rate'}]},
                      'valueQuantity': {'value': 18, 'unit': 'breaths/min'},
                      'effectiveDateTime': '2026-03-25T08:00:00Z'}},
        {'resource': {'resourceType': 'Observation', 'status': 'final',
                      'category': [{'coding': [{'code': 'vital-signs'}]}],
                      'code': {'coding': [{'code': '2708-6', 'display': 'Oxygen saturation (SpO2)'}]},
                      'valueQuantity': {'value': 96, 'unit': '%'},
                      'effectiveDateTime': '2026-03-25T08:00:00Z'}},
        {'resource': {'resourceType': 'Condition',
                      'clinicalStatus': {'coding': [{'code': 'active'}]},
                      'code': {'coding': [{'code': '44054006', 'display': 'Type 2 diabetes mellitus'}]},
                      'onsetDateTime': '2018-06-15'}},
        {'resource': {'resourceType': 'Condition',
                      'clinicalStatus': {'coding': [{'code': 'active'}]},
                      'code': {'coding': [{'code': '38341003', 'display': 'Essential hypertension'}]},
                      'onsetDateTime': '2015-01-20'}},
        {'resource': {'resourceType': 'MedicationRequest', 'status': 'active',
                      'medicationCodeableConcept': {'coding': [{'display': 'Metformin 500mg'}]}}},
        {'resource': {'resourceType': 'MedicationRequest', 'status': 'active',
                      'medicationCodeableConcept': {'coding': [{'display': 'Lisinopril 10mg'}]}}}
    ]
}

# Save and parse
bundle_path = '/content/sample_data/sample_patient_bundle.json'
with open(bundle_path, 'w') as f:
    json.dump(SAMPLE_BUNDLE, f, indent=2)

resources = parse_fhir_bundle(bundle_path)
demographics = extract_patient_demographics(resources['patients'][0])
vitals = extract_vitals(resources['observations'])
conditions = extract_conditions(resources['conditions'])
medications = extract_medications(resources['medications'])

print(f'Patient: {demographics["name"]}')
print(f'Vitals: {len(vitals)} | Conditions: {len(conditions)} | Medications: {len(medications)}')
print(f'\nVitals:\n{format_vitals_summary(vitals)}')

In [ ]:
# Generate admission note
from utils.encounter_state import load_encounter

enc = load_encounter(encounter_id)
soap = enc['stages']['consultation'].get('soap_note', {})
soap_text = '\n'.join(f'{k.upper()}: {v}' for k, v in soap.items() if v) or 'No SOAP note.'

demographics_str = f"Name: {demographics.get('name')}, DOB: {demographics.get('dob')}, Gender: {demographics.get('gender')}"
conditions_str = '\n'.join(f'- {c["display"]}' for c in conditions) or 'None'
medications_str = '\n'.join(f'- {m["display"]}' for m in medications) or 'None'
vitals_summary = format_vitals_summary(vitals)

print('Generating admission note...')
admission_note = generate_admission_note(
    demographics=demographics_str, soap_note=soap_text,
    vitals=vitals_summary, conditions=conditions_str,
    medications=medications_str, allergies='No known drug allergies (NKDA)',
    model=medgemma_model, processor=medgemma_processor
)

print('\n' + '='*60)
print('ADMISSION NOTE')
print('='*60)
print(admission_note)

# Save
encounter = update_stage(encounter_id, 'admission', {
    'fhir_patient_id': demographics.get('mrn', 'patient-001'),
    'vitals_history': vitals, 'conditions': conditions,
    'medications': medications, 'admission_note': admission_note
})
print(f'\nAdmission saved to encounter {encounter_id}')

---
## Section 4: Radiology — Medical Image → Report

In [ ]:
from utils.dicom_helpers import load_medical_image, download_sample_chest_xray
from utils.mcp_tools import analyze_medical_image
from IPython.display import display

# Get a sample medical image
sample_image_path = download_sample_chest_xray('/content/sample_data')
image, metadata = load_medical_image(sample_image_path)

print('Image metadata:')
for k, v in metadata.items():
    print(f'  {k}: {v}')

# Display
display(image.resize((300, 300)))

In [ ]:
# Build clinical context from prior stages
enc = load_encounter(encounter_id)
soap = enc['stages']['consultation'].get('soap_note', {})
admission = enc['stages']['admission'].get('admission_note', '')

clinical_context = ''
if isinstance(soap, dict):
    clinical_context += f"Assessment: {soap.get('assessment', 'N/A')}\n"
    clinical_context += f"Plan: {soap.get('plan', 'N/A')}\n"
if admission:
    clinical_context += f"\nAdmission Note:\n{str(admission)[:500]}"
if not clinical_context.strip():
    clinical_context = 'Patient with productive cough, fever. Rule out pneumonia.'

modality = metadata.get('modality', 'X-ray')
body_part = metadata.get('body_part', 'Chest')

print('Analyzing medical image with MedGemma...')
radiology_report = analyze_medical_image(
    image=image, clinical_context=clinical_context,
    modality=modality, body_part=body_part,
    model=medgemma_model, processor=medgemma_processor
)

print('\n' + '='*60)
print('RADIOLOGY REPORT')
print('='*60)
print(radiology_report)

# Save
encounter = update_stage(encounter_id, 'radiology', {
    'images': [{'orthanc_id': 'local', 'modality': modality, 'body_part': body_part, 'path': sample_image_path}],
    'reports': [{'findings': radiology_report, 'impression': '', 'recommendations': ''}]
})
print(f'\nRadiology saved to encounter {encounter_id}')

---
## Section 5: Discharge — Aggregate → Summary

In [ ]:
from utils.mcp_tools import generate_discharge_summary

enc = load_encounter(encounter_id)
consultation = enc['stages']['consultation']
admission_stage = enc['stages']['admission']
radiology_stage = enc['stages']['radiology']

# Format SOAP
soap = consultation.get('soap_note', {})
soap_text = ''
if isinstance(soap, dict):
    for s in ['subjective', 'objective', 'assessment', 'plan']:
        if soap.get(s):
            soap_text += f'**{s.title()}:** {soap[s]}\n\n'
if not soap_text.strip():
    soap_text = 'No consultation note.'

# Format vitals
vitals_trend = format_vitals_summary(admission_stage.get('vitals_history', []))

# Format radiology
rad_reports = ''
for i, r in enumerate(radiology_stage.get('reports', []), 1):
    rad_reports += f'Report {i}:\n{r.get("findings", "N/A")}\n\n'
if not rad_reports.strip():
    rad_reports = 'No radiology reports.'

admission_note_text = str(admission_stage.get('admission_note', 'No admission note.'))

print('Generating discharge summary...')
discharge_summary = generate_discharge_summary(
    soap_note=soap_text, admission_note=admission_note_text,
    vitals_trend=vitals_trend, radiology_reports=rad_reports,
    model=medgemma_model, processor=medgemma_processor, max_new_tokens=1536
)

print('\n' + '='*60)
print('DISCHARGE SUMMARY')
print('='*60)
print(discharge_summary)

In [ ]:
# ICD-10 code suggestions
icd10_suggestions = []
diagnosis_map = {
    'pneumonia': {'code': 'J18.9', 'description': 'Pneumonia, unspecified organism'},
    'type 2 diabetes': {'code': 'E11.9', 'description': 'Type 2 diabetes mellitus without complications'},
    'hypertension': {'code': 'I10', 'description': 'Essential (primary) hypertension'},
    'cough': {'code': 'R05.9', 'description': 'Cough, unspecified'},
    'fever': {'code': 'R50.9', 'description': 'Fever, unspecified'},
}

conditions_text = ' '.join(c.get('display', '') for c in admission_stage.get('conditions', []))
combined = (conditions_text + ' ' + str(soap.get('assessment', '')) + ' ' + discharge_summary).lower()

print('ICD-10 Code Suggestions:')
for keyword, code_info in diagnosis_map.items():
    if keyword in combined:
        icd10_suggestions.append(code_info)
        print(f'  {code_info["code"]}: {code_info["description"]}')

# Save discharge
meds_at_discharge = [m.get('display', '') for m in admission_stage.get('medications', []) if m.get('display')]
encounter = update_stage(encounter_id, 'discharge', {
    'summary': discharge_summary,
    'icd10_codes': icd10_suggestions,
    'medications_at_discharge': meds_at_discharge,
    'follow_up': 'Follow-up in 1 week. Return sooner if symptoms worsen.'
})
print(f'\nDischarge saved to encounter {encounter_id}')

---
## Section 6: Complete Encounter Record

In [ ]:
final = load_encounter(encounter_id)
stages = final['stages']

print('='*70)
print(f'COMPLETE ENCOUNTER: {encounter_id}')
print(f'Patient: {final["patient"]["name"]}')
print('='*70)

for stage_name in ['consultation', 'admission', 'radiology', 'discharge']:
    stage = stages[stage_name]
    status = 'COMPLETE' if stage.get('timestamp') else 'MISSING'
    print(f'\n--- {stage_name.upper()} [{status}] ---')
    print(f'  Timestamp: {stage.get("timestamp", "N/A")}')

# Key outputs
print(f'\n--- SOAP Assessment ---')
print(stages['consultation'].get('soap_note', {}).get('assessment', 'N/A')[:300])
print(f'\n--- Discharge Summary (first 500 chars) ---')
print(str(stages['discharge'].get('summary', 'N/A'))[:500])
print(f'\n--- ICD-10 Codes ---')
for code in stages['discharge'].get('icd10_codes', []):
    print(f'  {code["code"]}: {code["description"]}')

In [ ]:
# Export discharge report as Markdown
output_path = f'/content/encounters/{encounter_id}_discharge_report.md'
with open(output_path, 'w') as f:
    f.write(f'# Discharge Report: {encounter_id}\n\n')
    f.write(f'**Patient:** {final["patient"]["name"]}\n')
    f.write(f'**MRN:** {final["patient"]["mrn"]}\n')
    f.write(f'**DOB:** {final["patient"]["dob"]}\n\n---\n\n')
    f.write('## Consultation SOAP Note\n\n')
    s = stages['consultation'].get('soap_note', {})
    if isinstance(s, dict):
        for k, v in s.items():
            if v:
                f.write(f'**{k.title()}:** {v}\n\n')
    f.write('---\n\n## Admission Note\n\n')
    f.write(str(stages['admission'].get('admission_note', 'N/A')) + '\n\n')
    f.write('---\n\n## Radiology Reports\n\n')
    for r in stages['radiology'].get('reports', []):
        f.write(str(r.get('findings', 'N/A')) + '\n\n')
    f.write('---\n\n## Discharge Summary\n\n')
    f.write(str(stages['discharge'].get('summary', 'N/A')) + '\n\n')
    f.write('---\n\n## ICD-10 Codes\n\n')
    for c in stages['discharge'].get('icd10_codes', []):
        f.write(f'- **{c["code"]}**: {c["description"]}\n')

print(f'Discharge report: {output_path}')
print(f'Encounter JSON: /content/encounters/{encounter_id}.json')
print('\nFull pipeline complete!')